<div align="right"><i>Peter Norvig<br>October 2018<br>Type annotations 2026</i></div>

# Properly Ordered Card Hands

The 538 Riddler [presented](https://fivethirtyeight.com/features/who-will-capture-the-most-james-bonds/) this problem by Matt Ginsberg:
    
> *You play so many card games that you’ve developed a very specific organizational obsession. When you’re dealt your hand, you want to organize it such that the cards of a given suit are grouped together and, if possible, such that no suited groups of the same color are adjacent. (Numbers don’t matter to you.) Moreover, when you receive your randomly ordered hand, you want to achieve this organization with a single motion, moving only one adjacent block of cards to some other position in your hand, maintaining the original order of that block and other cards, except for that one move.*
>
> *Suppose you’re playing pitch, in which a hand has six cards. What are the odds that you can accomplish your obsessive goal? What about for another game, where a hand has N cards, somewhere between 1 and 13?*

# Brute Force 

The first thing to decide is how many `N`-card hands are there? If there are only a few, I could just enumerate them all and count how many are orderable. If there are a lot, I'll have to be more clever. The answer is (52 choose `N`), so we have:

- 6 cards: 20,358,520 hands
- 13 cards: 635,013,559,600 hands 

I could handle 6-card hands, but 600 billion is a lot of hands. Can we reduce the number of hands we have to deal with?

# Abstract Hands: Suits Only

The problem says *"Numbers don’t matter,"* so I can abstract away from "seven of spades" to just "a spade." Then there are only 4<sup>*N*</sup> abstract hands (for *N* &le; 13), so we have:

- 6 cards: 4,096 abstract hands
- 13 cards: 67,108,864 abstract hands

That's a big improvement! 

Now let's start coding:
- There are two red suits and two black suits, so I'll represent the four suits as `'rRbB'`.
- An abstract hand can be represented as a string of suits: `'rrBrbr'` is a 6-card hand. 
- `deals(N)` will return a dict of all possible abstract hands of length *N*, each mapped to the probability of the hand. 
- With actual hands, every hand has the same probability, because every card is equally likely to be the next card dealt. But with abstract hands, the probability of the next suit depends on how many cards of that suit have already been dealt. If I've already dealt the 12 cards `'rrrrrrrrrrrr'`, then the probability of the next card being an `'r'` is 1/40, and the probability of it being a `'b'` is 13/40. So as I build up the abstract hands, I'll need to keep track of the number of remaining cards of each suit.
- I'll use `Fraction` to get exact arithmetic and `cache` to avoid repeated computations.

In [1]:
from fractions import Fraction
from functools import cache
from typing import Iterable

type Hand = str
type Probability = Fraction | float

suits = 'rbRB'

@cache
def deals(N: int) -> dict[Hand, Probability]:
    """A dict of {hand: probability} for all hands of N cards."""
    if N == 0:
        return {'': Fraction(1)}
    else:
        P = deals(N - 1)
        return {hand + suit: P[hand] * (13 - hand.count(suit)) / (52 - len(hand))
                for hand in P
                for suit in suits}

In [2]:
deals(1)

{'r': Fraction(1, 4),
 'b': Fraction(1, 4),
 'R': Fraction(1, 4),
 'B': Fraction(1, 4)}

In [3]:
deals(2)

{'rr': Fraction(1, 17),
 'rb': Fraction(13, 204),
 'rR': Fraction(13, 204),
 'rB': Fraction(13, 204),
 'br': Fraction(13, 204),
 'bb': Fraction(1, 17),
 'bR': Fraction(13, 204),
 'bB': Fraction(13, 204),
 'Rr': Fraction(13, 204),
 'Rb': Fraction(13, 204),
 'RR': Fraction(1, 17),
 'RB': Fraction(13, 204),
 'Br': Fraction(13, 204),
 'Bb': Fraction(13, 204),
 'BR': Fraction(13, 204),
 'BB': Fraction(1, 17)}

Is that right? Yes it is. For `deals(1)`, each suit has probability 1/4. For `deals(2)`, the probability of `'BB'` is 1/17, beause the probability of the first `'B'` is 1/4, and when we deal the second card, one `'B'` is gone, so the probability is 12/51, so that simplifies to 1/4 &times; 12/51 = 1/17. The probability of `'BR'` is 1/4 &times; 13/51 = 13/204.

# More Abstraction: Collapsed Hands

Now for a second abstraction: an abstract hand can be *collapsed* by replacing a run of cards of the same suit with a single card, so that `'BBBBBrrrrBBBB'` collapses to `'BrB'`. We're interested in grouping cards of the same suit together, so any number of cards of the same suit is the same as a single card of that suit. 

In [4]:
type CollapsedHand = str

def collapse(hand: Hand) -> CollapsedHand:
    """Collapse multiple adjacent cards of the same suit into a single card of that suit."""
    return ''.join(hand[i] for i in range(len(hand)) if i == 0 or hand[i] != hand[i - 1])

In [5]:
assert collapse('BBBBBrrrrBBBB') == 'BrB'

# Properly Ordered Hands

A hand is considered properly `ordered` if *"the cards of a given suit are grouped together and, if possible, such that no suited groups of the same color are adjacent."* I was initially confused about the meaning of *"if possible";* Matt Ginsberg confirmed it means *"if it is possible to separate the colors in any number of moves"*, and thus that the hand `'BBBbbb'` is properly ordered, because the suits are all together, and it is not possible to separate the two black suits, while `'BBBbbR'` is not properly ordered, because the red card could be inserted between the two black runs.

In other words: a hand is properly ordered if and only if its collapsed hand is properly ordered, and a collapsed hand is properly ordered if each suit appears only once, and either all the colors are the same, or suits of the same color don't appear adjacent to each other.

In [6]:
def ordered(hand: Hand) -> bool:
    """Properly ordered if each suit run appears only once, and same color suits are not adjacent (or there is only one color)."""
    hand1 = collapse(hand)
    return once_each(hand1) and (len(set(colors(hand1))) == 1 or not adjacent_colors(hand1))

colors = str.lower # The color of both 'B' and 'b' is 'b'; the color of both 'R' and 'r' is 'r'
                                 
def adjacent_colors(hand: Hand) -> bool: 
    """Do two suits of the same color appear next to each other in a hand?"""
    return 'bb' in colors(hand) or 'rr' in colors(hand)

def once_each(hand: CollapsedHand) -> bool: 
    """Do all the suits in a collapsed hand appear just once each?"""
    return len(hand) == len(set(hand))

In [7]:
assert ordered('BBBbbb')
assert ordered('BBBRbb')
assert not ordered('BBBbbR') 
assert not ordered('BBBrBB') 

# Moving Cards to Make a Collapsed Hand Ordered

I'll say that a hand is `orderable` if any of the possible `moves` of a block of consecutive cards makes the hand `ordered`. It is more efficient to do this on collapsed hands. I'll throw a `cache` onto `orderable` so that it won't have to repeat computations.

To find all possible `moves`, consider every possible block of cards within a hand, and every way of placing the block into any position in the rest of the cards.

I'll define `orderable_probability(N)` to give the probability that a random *N*-card hand is orderable. 

In [8]:
@cache
def orderable(hand: CollapsedHand) -> bool: 
    """Can this collapsed hand be put into proper order in one move?"""
    return any(ordered(new_seq) for new_seq in moves(hand))

def moves(hand: Hand) -> Iterable[Hand]:
    """All ways of moving a block of cards to get a new hand."""
    N = len(hand)
    for i in range(N):
        for j in range(i + 1, N + 1):
            block = hand[i:j]
            rest  = hand[:i] + hand[j:]
            for k in range(len(rest) + 1):
                yield rest[:k] + block + rest[k:]

def orderable_probability(N: int) -> Probability:
    """What's the probability that an N-card hand is orderable?"""
    P = deals(N)
    return sum(P[hand] for hand in P if orderable(collapse(hand)))

# First Answer

Here's the answer for 6 cards:

In [9]:
orderable_probability(6) 

Fraction(51083, 83895)

And an easier-to-read answer for everything up to  6 cards:

In [10]:
def report(Ns: range) -> None:
    """Show the probability of orderability, for each N in Ns."""
    for N in Ns:
        P = orderable_probability(N)
        print('{:2}: {:8.3%} or {}'.format(N, float(P), P))
        
report(range(1, 7))

 1: 100.000% or 1
 2: 100.000% or 1
 3: 100.000% or 1
 4: 100.000% or 1
 5:  85.242% or 213019/249900
 6:  60.889% or 51083/83895


Let's make sure that each `deals(N)` covers everything: that I got all 4<sup>N</sup> hands, and that the probabilities sum to 1:

In [11]:
for N in range(7):
    assert len(deals(N)) == 4 ** N
    assert sum(deals(N).values()) == 1

# Getting to 13-Card Hands

So far so good, but if we want to get to 13-card hands, we would have to handle 4<sup>13</sup> = 67,108,864 `deals`, which would take multiple minutes. But I discovered two key properties that can speed things up:

1. **An orderable collapsed hand can have at most 7 characters.** We know that a properly ordered CollapsedHand can have at most 4 characters. But a move can reduce the number of characters by at most 3:  one can be reduced when we remove the block of cards (if the cards on either side of the block are the same), and up to two more can be reduced when we re-insert the block (if the left and/or right ends of the block match the surrounding suits). Here's an example of moving a block (in parens) to reduce the number of runs from 6 to 3:

       bRB(bR)B   =>   b(bR)RBB  =   bRB
       
3. **Adding cards to the end of an unorderable collapsed hand can't make it orderable.** To show that, take an unordered CollapsedHand, and see what happens if you take the extra suit and insert it anywhere in the CollapsedHand. If the CollapsedHand was unordered because it repeats a suit, adding a suit can't fix that. If it was unordered because suits of the same color are adjacent, then adding a suit of the other color *could* fix that: `'bBR'` could be fixed by inserting a `'r'` to get `'brBR'`. But here's the catch: `'bBR'` is not unorderable. And if we are going to insert a new suit between two others, that means that the original CollapsedHand must have had at most three suits (because when we add one, we can't get more than four suits in an ordered CollapsedHand), and *every* three-suit CollapsedHand is orderable.  

I'll define  `orderable_deals(N)` to return a {hand: probability\} dict of just the orderable hands, and
I'll redefine `orderable` and `orderable_probability` to take advantage of this.

In [12]:
@cache
def orderable_deals(N: int) -> dict[Hand, Probability]:
    """A dict of {hand: probability} for all orderable hands of length N."""
    if N == 0:
        return {'': Fraction(1)}
    else:
        P = orderable_deals(N - 1)
        return {hand + suit: P[hand] * (13 - hand.count(suit)) / (52 - len(hand))
                for hand in P
                for suit in suits
                if orderable(collapse(hand + suit))}
        
@cache
def orderable(hand: CollapsedHand) -> bool: 
    """Can this collapsed hand be put into proper order in one move?"""
    return len(hand) <= 7 and any(ordered(move) for move in moves(hand))

def orderable_probability(N: int) -> Probability:
    """What's the probability that an N-card hand is orderable?"""
    return sum(orderable_deals(N).values())

# Final Answer

We're finaly ready to go up to *N* = 13:

In [13]:
%time report(range(14))

 0: 100.000% or 1
 1: 100.000% or 1
 2: 100.000% or 1
 3: 100.000% or 1
 4: 100.000% or 1
 5:  85.242% or 213019/249900
 6:  60.889% or 51083/83895
 7:  37.321% or 33606799/90047300
 8:  20.185% or 29210911/144718875
 9:   9.861% or 133194539/1350709500
10:   4.432% or 367755247/8297215500
11:   1.859% or 22673450197/1219690678500
12:   0.736% or 1751664923/238130084850
13:   0.277% or 30785713171/11112737293000
CPU times: user 2.18 s, sys: 33.6 ms, total: 2.22 s
Wall time: 2.22 s


# Inspecting the Cache

Let's look at the cache for  `orderable(Hand)`:

In [14]:
orderable.cache_info()

CacheInfo(hits=1438512, misses=1540, maxsize=None, currsize=1540)

So `hits` say we looked at 1.4 million hands, but only 1540 were unique: distinct collapsed hands. And once we hit *N* = 7, we've seen all the collapsed hands we're ever going to see. From *N* = 8 and up, almost all the computation goes into computing the probability of each hand, and collapsing each hand, not into deciding orderability.

We also save a lot of space in the `deals(N)` caches. Instead of storing all 4<sup>13</sup> hands for `deals(13)`, the `report` above says that just 0.277% of the hands are orderable, so we reduced the storage requirements by a factor of 360.

# Unit Tests

To gain confidence in this project, here are some unit tests. Before declaring my answers definitively correct, I would want a lot more tests, and some independent code reviews.

In [15]:
def test():
    assert deals(1) == {'B': 1/4, 'R': 1/4, 'b': 1/4, 'r': 1/4}
    assert ordered('BBBBBrrrrBBBB') is False
    assert ordered('BBBBBrrrrRRRR') is False
    assert ordered('BBBbbr') is False # Bb
    assert ordered('BBBbbrB') is False # two B's
    assert ordered('BBBbbb') 
    assert ordered('BBBbbbB') is False # two B's
    assert ordered('BBBBBrrrrbbbb')
    assert colors('BBBBBrrrrbbbb') == 'bbbbbrrrrbbbb'
    assert once_each('Bb')
    assert once_each('BbRr')
    assert once_each('BbRB') is False
    assert adjacent_colors('BbR')
    assert adjacent_colors('Brb') is False
    assert collapse('BBBBBrrrrBBBB') == 'BrB'
    assert collapse('brBBrrRR') == 'brBrR'
    assert collapse('bbbbBBBrrr') == 'bBr'
    assert set(moves('bRb')) == {'Rbb', 'bRb', 'bbR'}
    assert set(moves('bRB')) == {'BbR', 'RBb', 'RbB', 'bBR', 'bRB'}
    assert orderable('bBr') # move 'r' between 'bB'
    assert orderable('bBrbRBr') # move 'bRB' after first 'b' to get 'bbRBBrr'
    assert orderable('bBrbRBrb') is False
    return True

test()

True